============================================================
SPARK MEMORY & CONFIGURATIONS - Lead Engineer Interview
============================================================
Topic: Memory Management, GC, Serialization, All Configs
============================================================


EXECUTOR MEMORY BREAKDOWN:

┌────────────────────────────────────────────────────────────────────┐
│                  spark.executor.memory (e.g., 8g)                   │
├────────────────────────────────────────────────────────────────────┤
│  ┌───────────────────────────────────────────────────────────────┐ │
│  │         UNIFIED MEMORY (spark.memory.fraction = 0.6)          │ │
│  │                        = 8g × 0.6 = 4.8GB                     │ │
│  ├───────────────────────────┬───────────────────────────────────┤ │
│  │    STORAGE MEMORY         │      EXECUTION MEMORY             │ │
│  │  (spark.memory.storage    │    (Joins, Shuffles, Sorts)       │ │
│  │    Fraction = 0.5)        │                                   │ │
│  │    = 4.8g × 0.5 = 2.4GB   │    = 4.8g × 0.5 = 2.4GB          │ │
│  │                           │                                   │ │
│  │  - Cached RDDs/DFs        │  - Shuffle buffers                │ │
│  │  - Broadcast variables    │  - Join hash tables               │ │
│  │                           │  - Aggregation buffers            │ │
│  │                           │                                   │ │
│  │  CAN BORROW FROM ─────────┼────────► CAN BORROW FROM          │ │
│  │  EXECUTION IF FREE        │          STORAGE (evicts cache)   │ │
│  └───────────────────────────┴───────────────────────────────────┘ │
├────────────────────────────────────────────────────────────────────┤
│              USER MEMORY (1 - 0.6 = 0.4) = 3.2GB                    │
│                                                                     │
│  - User data structures                                             │
│  - Spark internal metadata                                          │
│  - UDF variables                                                    │
└────────────────────────────────────────────────────────────────────┘

PLUS: spark.executor.memoryOverhead (OFF-HEAP)
┌────────────────────────────────────────────────────────────────────┐
│           MEMORY OVERHEAD (default: max(384MB, 10% of executor))    │
│                                                                     │
│  - JVM overhead (thread stacks, NIO buffers)                        │
│  - Python worker memory (PySpark)                                   │
│  - Container overhead                                               │
│  - Off-heap storage                                                 │
└────────────────────────────────────────────────────────────────────┘

TOTAL CONTAINER SIZE = spark.executor.memory + spark.executor.memoryOverhead



KEY MEMORY SETTINGS:

# Executor memory (on-heap)
spark.executor.memory = 8g

# Off-heap overhead (CRITICAL for PySpark!)
spark.executor.memoryOverhead = 2g  # Default: max(384MB, 10%)

# Unified memory pool
spark.memory.fraction = 0.6  # 60% for storage + execution

# Storage vs Execution split
spark.memory.storageFraction = 0.5  # Initial 50-50 split

# Off-heap memory (outside JVM)
spark.memory.offHeap.enabled = true
spark.memory.offHeap.size = 2g

# Driver memory
spark.driver.memory = 4g
spark.driver.memoryOverhead = 1g

WHEN TO INCREASE memoryOverhead:
1. PySpark jobs (Python workers need memory)
2. Native libraries (pandas, numpy)
3. Many concurrent tasks
4. Complex UDFs
5. "Container killed by YARN" errors



GC ISSUES IN SPARK:
- Long GC pauses → task timeouts
- Full GC → "stop the world"
- Memory pressure → frequent GC

GC CONFIGURATIONS:
spark.executor.extraJavaOptions = "-XX:+UseG1GC -XX:MaxGCPauseMillis=200"

COMMON GC OPTIONS:
-XX:+UseG1GC                    # G1 garbage collector (recommended)
-XX:MaxGCPauseMillis=200        # Target pause time
-XX:InitiatingHeapOccupancy=45  # Start GC earlier
-XX:+PrintGCDetails             # Debug logging
-XX:+PrintGCTimeStamps

DIAGNOSING GC ISSUES:
1. Check Spark UI > Executors > GC Time
2. If GC Time > 10% of task time → problem
3. Solutions:
   - Increase executor memory
   - Reduce partition size
   - Use off-heap memory
   - Cache less data

MEMORY PRESSURE SYMPTOMS:
- Frequent GC pauses
- Container killed by YARN
- OutOfMemoryError
- Slow tasks, fast GC time



SERIALIZATION: Converting objects to bytes for transfer

JAVA SERIALIZATION (Default):
- Works with any Serializable class
- Slow and verbose
- Large serialized size

KRYO SERIALIZATION (Recommended):
- 10x faster than Java
- Smaller serialized size
- Requires registration for best performance

CONFIGURATION:
spark.serializer = org.apache.spark.serializer.KryoSerializer
spark.kryo.registrationRequired = false  # Set true for production
spark.kryo.classesToRegister = com.example.MyClass1,com.example.MyClass2

CODE TO REGISTER:
from pyspark import SparkConf
conf = SparkConf()
conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
conf.registerKryoClasses([MyClass1, MyClass2])

WHEN SERIALIZATION MATTERS:
1. Shuffle (data between stages)
2. Caching with MEMORY_ONLY_SER
3. Broadcasting data
4. RDD closures (UDFs)

PYSPARK SERIALIZATION:
- Uses Pickle by default
- CloudPickle for closures
- Arrow for Pandas UDFs (fastest!)



PARALLELISM & PARTITIONS:
spark.default.parallelism = 200           # RDD parallelism
spark.sql.shuffle.partitions = 200        # DataFrame shuffle partitions
spark.sql.files.maxPartitionBytes = 128MB # Max partition size for file read
spark.sql.files.minPartitionNum = 1       # Min partitions for file read

SHUFFLE:
spark.shuffle.compress = true
spark.shuffle.file.buffer = 64k
spark.shuffle.spill.compress = true
spark.sql.shuffle.partitions = 200

BROADCAST:
spark.sql.autoBroadcastJoinThreshold = 10MB  # Auto broadcast if smaller
spark.broadcast.compress = true

JOIN STRATEGIES:
spark.sql.join.preferSortMergeJoin = true
spark.sql.autoBroadcastJoinThreshold = 10485760  # 10MB

ADAPTIVE QUERY EXECUTION (Spark 3.0+):
spark.sql.adaptive.enabled = true
spark.sql.adaptive.coalescePartitions.enabled = true
spark.sql.adaptive.skewJoin.enabled = true
spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 256MB

DYNAMIC ALLOCATION:
spark.dynamicAllocation.enabled = true
spark.dynamicAllocation.minExecutors = 2
spark.dynamicAllocation.maxExecutors = 100
spark.dynamicAllocation.executorIdleTimeout = 60s
spark.dynamicAllocation.schedulerBacklogTimeout = 1s

SPECULATION:
spark.speculation = false
spark.speculation.multiplier = 1.5
spark.speculation.quantile = 0.75

NETWORK:
spark.network.timeout = 120s
spark.executor.heartbeatInterval = 10s

STORAGE:
spark.storage.level = MEMORY_AND_DISK_SER
spark.rdd.compress = true

SQL OPTIMIZATIONS:
spark.sql.cbo.enabled = true                    # Cost-based optimizer
spark.sql.cbo.joinReorder.enabled = true
spark.sql.inMemoryColumnarStorage.compressed = true



WHAT IS SPILL?
When execution memory is full, data spills to disk.

WHERE SPILL HAPPENS:
1. Shuffle Write: Map output buffers full
2. Shuffle Read: Reduce-side merge
3. Aggregation: Hash map too large
4. Sort: Sort buffer overflow

SPILL CONFIGURATION:
spark.shuffle.spill = true (default)
spark.shuffle.spill.compress = true
spark.shuffle.file.buffer = 64k

MONITORING SPILL:
- Spark UI > Stages > Shuffle Spill (Memory/Disk)
- High spill = memory pressure

REDUCING SPILL:
1. Increase spark.executor.memory
2. Reduce partition size (more partitions)
3. Increase spark.memory.fraction
4. Use broadcast for small tables
5. Filter data earlier

SPILL LOCATIONS:
spark.local.dir = /tmp/spark  # Comma-separated list for multiple disks
# Use fast SSDs, multiple directories for parallelism



PARTITION SIZE GUIDELINES:
- Target: 128MB - 1GB per partition
- Too small: Task scheduling overhead
- Too large: Memory pressure, long GC, uneven load

CALCULATING PARTITIONS:
data_size_gb = 100  # 100GB input
target_partition_mb = 256
num_partitions = (data_size_gb * 1024) / target_partition_mb  # = 400

HOW TO SET:
# For DataFrames (shuffle output)
spark.conf.set("spark.sql.shuffle.partitions", 400)

# For specific operations
df.repartition(400)
df.coalesce(100)

REPARTITION vs COALESCE:
repartition(n): Full shuffle, can increase or decrease
coalesce(n): No shuffle, can only decrease, may be uneven

REPARTITION BY COLUMN:
df.repartition("key")  # Same keys go to same partition
df.repartition(100, "key")  # Control number + key

PARTITION FILE OUTPUT:
df.repartition(10).write.parquet("output")  # 10 files
df.coalesce(1).write.csv("output")  # 1 file (careful with large data!)



map(): Function applied to EACH ELEMENT
mapPartitions(): Function applied to EACH PARTITION

WHEN TO USE mapPartitions:
1. Expensive initialization (DB connection, HTTP session)
2. Batch operations across partition
3. External API calls
4. Loading ML models

EXAMPLE:
# map: Creates connection for EVERY row (BAD)
rdd.map(lambda x: query_db(x))  # N connections

# mapPartitions: Creates connection ONCE per partition (GOOD)
def process_partition(iterator):
    conn = create_db_connection()  # Once per partition
    for row in iterator:
        yield query_db(conn, row)
    conn.close()

rdd.mapPartitions(process_partition)  # P connections (P << N)

DATAFRAME EQUIVALENT:
df.mapInPandas(process_func, schema)
df.rdd.mapPartitions(process_func)

MEMORY CONSIDERATION:
mapPartitions loads entire partition into memory
For large partitions, use iterator pattern (yield)



BUCKETING: Pre-shuffle optimization for repeated joins

HOW IT WORKS:
- Data partitioned by hash of bucket column
- Same number of buckets = no shuffle on join

CREATE BUCKETED TABLE:
df.write     .bucketBy(16, "join_key")     .sortBy("join_key")     .saveAsTable("bucketed_table")

BENEFITS:
1. No shuffle for joins on bucket column
2. Faster aggregations on bucket column
3. Sorted buckets enable merge join

WHEN TO USE:
- Frequent joins on same column
- Large tables joined repeatedly
- Stable join key

REQUIREMENTS:
- Must use saveAsTable (not write.parquet)
- Both tables must have same bucket count
- Join on bucket column
- spark.sql.sources.bucketing.enabled = true



STORAGE LEVELS:

| Level              | Heap | Off-Heap | Disk | Serialized |
|--------------------|------|----------|------|------------|
| MEMORY_ONLY        | ✓    |          |      |            |
| MEMORY_AND_DISK    | ✓    |          | ✓    |            |
| MEMORY_ONLY_SER    | ✓    |          |      | ✓          |
| MEMORY_AND_DISK_SER| ✓    |          | ✓    | ✓          |
| DISK_ONLY          |      |          | ✓    |            |
| OFF_HEAP           |      | ✓        |      | ✓          |

RECOMMENDATIONS:
1. Small data: MEMORY_ONLY
2. Medium data: MEMORY_AND_DISK
3. Large data: MEMORY_AND_DISK_SER (saves memory, CPU cost)
4. Avoid GC: OFF_HEAP

CACHE vs PERSIST:
cache() = persist(StorageLevel.MEMORY_AND_DISK)
persist(level) = choose storage level

CACHE EVICTION:
- LRU (Least Recently Used)
- Execution memory can evict storage if needed

WHEN TO CACHE:
✓ DataFrame used multiple times
✓ After expensive computation
✓ Interactive analysis

WHEN NOT TO CACHE:
✗ One-time use
✗ Small data (faster to recompute)
✗ Memory pressure already high


In [ ]:
if __name__ == "__main__":
    from pyspark.sql import SparkSession
    
    spark = SparkSession.builder \
        .appName("MemoryDemo") \
        .master("local[*]") \
        .config("spark.executor.memory", "2g") \
        .config("spark.executor.memoryOverhead", "512m") \
        .config("spark.memory.fraction", "0.6") \
        .config("spark.memory.storageFraction", "0.5") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.sql.shuffle.partitions", "50") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()
    
    # Print key configurations
    print("Key Memory Configurations:")
    print(f"  executor.memory: {spark.conf.get('spark.executor.memory', 'default')}")
    print(f"  memory.fraction: {spark.conf.get('spark.memory.fraction')}")
    print(f"  shuffle.partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
    print(f"  serializer: {spark.conf.get('spark.serializer')}")
    
    spark.stop()
